Clinical Trial Radar - Stage 1

Definirea problemei si analiza datelor de intrare

Acest notebook este pentru prima etapa a proiectului. Scopul lui este sa explice problema, sa prezinte datele folosite si sa faca o analiza initiala a datasetului obtinut din ClinicalTrials.gov.

Notebook-ul poate fi rulat din radacina proiectului sau din folderul docs. Codul cauta automat folderul data/raw.


1. Problema abordata

Studiile clinice pot fi greu de planificat deoarece informatiile despre studii similare sunt imprastiate in registre publice si trebuie comparate manual. Pentru o echipa care vrea sa porneasca un nou studiu, este important sa stie unde exista deja multe studii active, unde s-au finalizat studii in trecut si cat dureaza in mod obisnuit un studiu.

Proiectul Clinical Trial Radar propune un sistem inteligent care analizeaza automat studiile clinice pentru o anumita boala si o anumita faza. Sistemul foloseste date publice din ClinicalTrials.gov si genereaza indicatori utili pentru analiza de fezabilitate.

Intrebarea principala a proiectului este:

Pentru o anumita boala si o anumita faza, cat de aglomerat este peisajul studiilor clinice si ce tari par mai potrivite pentru recrutare?


2. Inputuri si outputuri

Inputuri:
- boala analizata, de exemplu diabetes, breast cancer sau covid-19;
- faza studiului, de exemplu Phase 1, Phase 2 sau Phase 3;
- numarul maxim de studii care vor fi descarcate;
- optional, tara sau regiunea de interes.

Outputuri:
- tabel cu studii relevante;
- distributie pe statusuri, tari si ani;
- grafic de trend;
- harta pe tari;
- metrica de aglomerare pentru studiile active/recruiting;
- estimare pentru durata studiilor;
- feasibility snapshot in format HTML sau PPTX.


3. Sursa datelor

Datele folosite provin din ClinicalTrials.gov, un registru public de studii clinice. Aplicatia descarca datele cu ajutorul unui modul de ingestie si le salveaza in folderul data/raw sub forma de fisiere CSV.

In aceasta etapa analizam fisierele CSV deja descarcate in proiect.


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 120)

def find_project_root():
    current = Path.cwd().resolve()
    for path in [current] + list(current.parents):
        if (path / "data" / "raw").exists():
            return path
    return current

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data" / "raw"

print("Project root:", PROJECT_ROOT)
print("Data folder:", DATA_DIR)
print("Data folder exists:", DATA_DIR.exists())

4. Incarcarea datelor

Incarcam toate fisierele CSV din data/raw. Fiecare fisier primeste o coloana suplimentara SourceFile, ca sa stim din ce fisier provine fiecare rand.


In [ ]:
csv_files = sorted(DATA_DIR.glob("*.csv"))

print("Numar fisiere CSV gasite:", len(csv_files))
for file in csv_files:
    print("-", file.name)

dataframes = []

for file in csv_files:
    try:
        temp = pd.read_csv(file)
        temp["SourceFile"] = file.name
        dataframes.append(temp)
    except Exception as error:
        print("Nu am putut citi fisierul:", file.name)
        print(error)

if dataframes:
    df = pd.concat(dataframes, ignore_index=True)
else:
    df = pd.DataFrame()

print("Numar total de randuri:", len(df))
print("Numar total de coloane:", len(df.columns))
df.head()

5. Coloane importante

Pentru analiza de fezabilitate ne intereseaza in special urmatoarele campuri:

- NCTId: identificatorul studiului clinic;
- BriefTitle: titlul scurt al studiului;
- OverallStatus: statusul studiului, de exemplu Recruiting sau Completed;
- Phase: faza studiului clinic;
- StartDate: data de inceput;
- CompletionDate: data de finalizare;
- LocationCountry: tara sau tarile in care are loc studiul;
- LocationCity: orasul sau orasele in care are loc studiul;
- EnrollmentCount: numarul de participanti;
- EligibilityCriteria: criterii de includere si excludere;
- PrimaryOutcomes: obiectivele principale ale studiului.


In [ ]:
print("Coloanele disponibile:")
for col in df.columns:
    print("-", col)

6. Dimensiunea datasetului

Verificam cate studii avem in total si cate studii unice exista pe baza identificatorului NCTId.


In [ ]:
if not df.empty:
    total_rows = len(df)
    unique_trials = df["NCTId"].nunique() if "NCTId" in df.columns else "coloana lipsa"
    print("Randuri totale:", total_rows)
    print("Studii unice:", unique_trials)
else:
    print("Nu exista date incarcate.")

7. Valori lipsa

Datele din registre publice pot avea campuri lipsa. De exemplu, unele studii nu au data completa de finalizare, locatie completa sau criterii de eligibilitate detaliate.

Mai jos calculam cate valori lipsa exista pentru coloanele importante.


In [ ]:
important_columns = [
    "NCTId",
    "BriefTitle",
    "OverallStatus",
    "Phase",
    "StartDate",
    "CompletionDate",
    "LocationCountry",
    "LocationCity",
    "EnrollmentCount",
    "EligibilityCriteria",
    "PrimaryOutcomes",
]

existing_columns = [col for col in important_columns if col in df.columns]

if existing_columns:
    missing_summary = (
        df[existing_columns]
        .isna()
        .sum()
        .reset_index()
    )
    missing_summary.columns = ["Column", "MissingValues"]
    missing_summary["MissingPercent"] = (missing_summary["MissingValues"] / len(df) * 100).round(2)
    display(missing_summary)
else:
    print("Nu s-au gasit coloanele importante in dataset.")

8. Distributia studiilor dupa status

Statusul studiului este important pentru ca arata daca studiile sunt active, in recrutare, finalizate sau retrase. Pentru metrica de aglomerare ne intereseaza in special studiile active sau in recrutare.


In [ ]:
if "OverallStatus" in df.columns and not df.empty:
    status_counts = df["OverallStatus"].fillna("UNKNOWN").value_counts()
    display(status_counts.to_frame("NumberOfStudies"))

    plt.figure(figsize=(10, 5))
    status_counts.plot(kind="bar")
    plt.title("Distributia studiilor dupa status")
    plt.xlabel("Status")
    plt.ylabel("Numar studii")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()
else:
    print("Coloana OverallStatus lipseste.")

9. Distributia studiilor pe faze

Verificam fazele studiilor existente in date. Uneori un studiu poate avea mai multe faze mentionate, de exemplu PHASE2; PHASE3.


In [ ]:
if "Phase" in df.columns and not df.empty:
    phase_counts = df["Phase"].fillna("UNKNOWN").value_counts()
    display(phase_counts.to_frame("NumberOfStudies"))

    plt.figure(figsize=(10, 5))
    phase_counts.plot(kind="bar")
    plt.title("Distributia studiilor dupa faza")
    plt.xlabel("Faza")
    plt.ylabel("Numar studii")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()
else:
    print("Coloana Phase lipseste.")

10. Distributia geografica

Pentru analiza de fezabilitate este important sa vedem in ce tari exista cele mai multe studii. Daca o tara are multe studii active, poate exista competitie mai mare pentru recrutarea pacientilor.

Unele randuri contin mai multe tari separate prin punct si virgula. De aceea transformam aceste valori in liste si le explodam.


In [ ]:
def split_semicolon_values(value):
    if pd.isna(value):
        return []
    return [item.strip() for item in str(value).split(";") if item.strip()]

if "LocationCountry" in df.columns and not df.empty:
    countries = df[["NCTId", "OverallStatus", "LocationCountry"]].copy()
    countries["Country"] = countries["LocationCountry"].apply(split_semicolon_values)
    countries = countries.explode("Country")
    countries = countries[countries["Country"].notna() & (countries["Country"] != "")]

    country_counts = countries["Country"].value_counts().head(10)
    display(country_counts.to_frame("NumberOfTrialLocations"))

    plt.figure(figsize=(10, 5))
    country_counts.plot(kind="bar")
    plt.title("Top 10 tari dupa numarul de locatii/studii")
    plt.xlabel("Tara")
    plt.ylabel("Numar aparitii")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()
else:
    print("Coloana LocationCountry lipseste.")

11. Trend in timp

Pentru trend folosim StartDate si extragem anul de inceput. Acest grafic arata cum a evoluat numarul de studii in timp.


In [ ]:
if "StartDate" in df.columns and not df.empty:
    trend_df = df.copy()
    trend_df["ParsedStartDate"] = pd.to_datetime(trend_df["StartDate"], errors="coerce")
    trend_df["StartYear"] = trend_df["ParsedStartDate"].dt.year

    yearly_counts = trend_df["StartYear"].dropna().astype(int).value_counts().sort_index()
    display(yearly_counts.to_frame("NumberOfStudies"))

    plt.figure(figsize=(10, 5))
    yearly_counts.plot(kind="line", marker="o")
    plt.title("Numar de studii incepute pe an")
    plt.xlabel("An")
    plt.ylabel("Numar studii")
    plt.tight_layout()
    plt.show()
else:
    print("Coloana StartDate lipseste.")

12. Trend pe statusuri importante

Pentru cerinta proiectului este util sa comparam studiile Recruiting cu cele Completed. Daca datele contin aceste statusuri, generam un tabel si un grafic pentru evolutia lor in timp.


In [ ]:
if {"StartDate", "OverallStatus"}.issubset(df.columns) and not df.empty:
    trend_status = df.copy()
    trend_status["ParsedStartDate"] = pd.to_datetime(trend_status["StartDate"], errors="coerce")
    trend_status["StartYear"] = trend_status["ParsedStartDate"].dt.year
    trend_status["OverallStatusClean"] = trend_status["OverallStatus"].fillna("UNKNOWN").str.upper()

    selected_statuses = ["RECRUITING", "COMPLETED"]
    filtered = trend_status[trend_status["OverallStatusClean"].isin(selected_statuses)]

    if not filtered.empty:
        pivot = (
            filtered
            .pivot_table(
                index="StartYear",
                columns="OverallStatusClean",
                values="NCTId",
                aggfunc="count",
                fill_value=0
            )
            .sort_index()
        )

        display(pivot)

        plt.figure(figsize=(10, 5))
        for column in pivot.columns:
            plt.plot(pivot.index, pivot[column], marker="o", label=column)
        plt.title("Trend Recruiting vs Completed")
        plt.xlabel("An")
        plt.ylabel("Numar studii")
        plt.legend()
        plt.tight_layout()
        plt.show()
    else:
        print("Nu exista statusuri Recruiting sau Completed in datele incarcate.")
else:
    print("Coloanele necesare lipsesc.")

13. Aglomerare pentru recrutare

O varianta simpla pentru crowdedness este numarul de studii active sau in recrutare din fiecare tara. Cu cat o tara are mai multe studii active, cu atat poate exista competitie mai mare pentru recrutarea participantilor.


In [ ]:
active_statuses = {
    "RECRUITING",
    "ACTIVE_NOT_RECRUITING",
    "NOT_YET_RECRUITING",
    "ENROLLING_BY_INVITATION"
}

if {"OverallStatus", "LocationCountry"}.issubset(df.columns) and not df.empty:
    active_df = df.copy()
    active_df["OverallStatusClean"] = active_df["OverallStatus"].fillna("UNKNOWN").str.upper()
    active_df = active_df[active_df["OverallStatusClean"].isin(active_statuses)]

    active_countries = active_df[["NCTId", "OverallStatus", "LocationCountry"]].copy()
    active_countries["Country"] = active_countries["LocationCountry"].apply(split_semicolon_values)
    active_countries = active_countries.explode("Country")
    active_countries = active_countries[active_countries["Country"].notna() & (active_countries["Country"] != "")]

    crowdedness = active_countries["Country"].value_counts().head(10)
    display(crowdedness.to_frame("ActiveOrRecruitingStudies"))

    plt.figure(figsize=(10, 5))
    crowdedness.plot(kind="bar")
    plt.title("Top 10 tari dupa aglomerarea studiilor active")
    plt.xlabel("Tara")
    plt.ylabel("Studii active / in recrutare")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()
else:
    print("Coloanele necesare lipsesc.")

14. Durata studiilor

Pentru estimarea duratei folosim diferenta dintre StartDate si CompletionDate. Rezultatul este aproximativ deoarece unele date pot fi incomplete sau lipsa.


In [ ]:
if {"StartDate", "CompletionDate"}.issubset(df.columns) and not df.empty:
    duration_df = df.copy()
    duration_df["ParsedStartDate"] = pd.to_datetime(duration_df["StartDate"], errors="coerce")
    duration_df["ParsedCompletionDate"] = pd.to_datetime(duration_df["CompletionDate"], errors="coerce")
    duration_df["DurationDays"] = (duration_df["ParsedCompletionDate"] - duration_df["ParsedStartDate"]).dt.days
    duration_df = duration_df[duration_df["DurationDays"].notna() & (duration_df["DurationDays"] >= 0)]

    if not duration_df.empty:
        duration_summary = duration_df["DurationDays"].describe().to_frame("DurationDays")
        display(duration_summary)

        print("Durata mediana in zile:", round(duration_df["DurationDays"].median(), 2))
        print("Durata mediana in luni:", round(duration_df["DurationDays"].median() / 30.44, 2))

        plt.figure(figsize=(10, 5))
        duration_df["DurationDays"].plot(kind="hist", bins=20)
        plt.title("Distributia duratelor studiilor")
        plt.xlabel("Durata in zile")
        plt.ylabel("Numar studii")
        plt.tight_layout()
        plt.show()
    else:
        print("Nu exista suficiente date valide pentru calculul duratei.")
else:
    print("Coloanele StartDate si CompletionDate lipsesc.")

15. Observatii initiale

Completati aceasta sectiune dupa ce rulati notebook-ul.

Exemplu de observatii care pot fi scrise:
- datasetul contine mai multe studii pentru diabetes, in special pentru Phase 2;
- cele mai frecvente statusuri sunt Completed si Recruiting;
- Statele Unite apar frecvent ca locatie pentru studii;
- unele coloane au valori lipsa, mai ales pentru locatie sau descrieri detaliate;
- datele sunt potrivite pentru metrici de tip trend, crowdedness si cycle time.


16. Concluzie Stage 1

In prima etapa am definit problema, inputurile si outputurile sistemului si am analizat datele de intrare. Datele din ClinicalTrials.gov contin suficiente informatii pentru a construi indicatori de fezabilitate, cum ar fi distributia pe tari, evolutia in timp, statusul studiilor si durata estimata.

In etapa urmatoare, aceste date vor fi folosite pentru dezvoltarea agentului AI, pentru generarea automata a snapshotului si pentru recomandari privind selectia tarilor.
